In [ ]:
import pandas as pd
from itables import show

data_raw = pd.read_parquet("data.parquet")

In [1]:
!pip install pycodestyle

In [ ]:
print(data_raw.shape)
show(data_raw.iloc[1000])

(2643425, 70)


Loading ITables v2.7.0 from the internet... (need help?)


In [ ]:
mixture_counts = data_raw["components"].value_counts()
print(mixture_counts.head(50))

components
germanium iron lithium oxide (Ge2FeLiO6), germanium iron sodium oxide (Ge2FeNaO6)    15941
decane                                                                               10684
pentaerythritol tetrahexanoate                                                       10246
pentaerythritol tetraheptanoate                                                       7686
ammonium nitrate, water, calcium nitrate                                              7444
1,1,1,2,3,3,3-heptafluoropropane                                                      7428
water                                                                                 6530
pentaerythritol tetrapentanoate                                                       5403
ethanol, water                                                                        5038
pentaerythritol tetrabutyrate                                                         4770
pentaerythritol tetraoctanoate                                                 

In [ ]:
tmp = data_raw.copy()
tmp["comp_list"] = tmp["components"].astype(str).str.lower().str.split(r",\s+")
tmp["ncomp"] = tmp["comp_list"].apply(lambda xs: len(set(xs)))
tmp["mix_key"] = tmp["comp_list"].apply(lambda xs: "|".join(sorted(set(xs))))

binary = tmp[tmp["ncomp"] == 2]

summary = (
    binary.groupby("mix_key")["source_file"].nunique().sort_values(ascending=False)
)

print("Unique binary mixtures:", binary["mix_key"].nunique())
print(binary["mix_key"].value_counts().head(20))

print("Number of source files per binary mixtures:", summary.shape[0], " mixtures")
print(summary.head(30))  # top 30 by source files

Unique binary mixtures: 28044
mix_key
germanium iron lithium oxide (ge2felio6)|germanium iron sodium oxide (ge2fenao6)    15941
ethanol|water                                                                        5121
carbon dioxide|methanol                                                              4546
triethylene glycol|water                                                             3638
methanol|water                                                                       3617
difluoromethane|pentafluoroethane                                                    3195
carbon dioxide|water                                                                 3081
2-aminoacetic acid|water                                                             3019
propan-2-ol|water                                                                    2967
propan-1-ol|water                                                                    2789
diisopropyl ether|propan-2-ol                                 

In [ ]:
binary.shape

(1359312, 73)

In [ ]:
rows_per_file = binary.groupby("source_file").size().sort_values()

min_rows = int(rows_per_file.min())

print("Minimum #rows in a file (among fixed binaries):", min_rows)

print("\nFile(s) with that minimum:")

min_files = rows_per_file[rows_per_file == min_rows].index.tolist()

print("\n".join(min_files))

Minimum #rows in a file (among fixed binaries): 1

File(s) with that minimum:
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.jct.2013.02.011.xml
C:\Users\harry\.thermoml\extracted_xml\10.1021\je900929g.xml
C:\Users\harry\.thermoml\extracted_xml\10.1021\je900936t.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2016.08.010.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2017.05.023.xml
C:\Users\harry\.thermoml\extracted_xml\10.1021\je0603628.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.jct.2006.03.003.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.jct.2006.01.006.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2017.03.022.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2017.04.013.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2015.08.006.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.jct.2016.05.017.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.tca.2015.03.027.xml
C:\Users\harry\.thermoml\extracted_xml\10.1016\j.

In [ ]:
MIN_ROWS = 5

rows_per_file = binary.groupby("source_file").size()
keep_files = rows_per_file[rows_per_file >= MIN_ROWS].index

binary_5plusperfile = binary[binary["source_file"].isin(keep_files)].copy()

print("Files before:", rows_per_file.size)
print("Files after :", len(keep_files))
print("Rows before :", len(binary))
print("Rows after  :", len(binary_5plusperfile))

Files before: 8088
Files after : 7786
Rows before : 1359312
Rows after  : 1358518


In [ ]:
binary_mixture_counts = binary_5plusperfile["mix_key"].value_counts()
print(binary_mixture_counts)

mix_key
germanium iron lithium oxide (ge2felio6)|germanium iron sodium oxide (ge2fenao6)    15941
ethanol|water                                                                        5116
carbon dioxide|methanol                                                              4546
triethylene glycol|water                                                             3635
methanol|water                                                                       3617
                                                                                    ...  
2,3,5-trimethyl-1-octylpyridinium 1,1,1-trifluoromethanesulfonate|water                 1
3,5-dimethyl-1-octylpyridinium dicyanamide|water                                        1
5-ethyl-2-methyl-1-octylpyridinium dicyanamide|water                                    1
3,5-dimethyl-1-octylpyridinium thiocyanate|water                                        1
2,3-dimethyl-1-octylpyridinium thiocyanate|water                                        1
Na

In [ ]:
new_binary_mixture_counts = binary_mixture_counts[binary_mixture_counts > 50]
print(new_binary_mixture_counts)

mix_key
germanium iron lithium oxide (ge2felio6)|germanium iron sodium oxide (ge2fenao6)    15941
ethanol|water                                                                        5116
carbon dioxide|methanol                                                              4546
triethylene glycol|water                                                             3635
methanol|water                                                                       3617
                                                                                    ...  
1-undecanol|acetonitrile                                                               51
cyclohexylamine|toluene                                                                51
2-methylbutan-2-ol|ethyl acetate                                                       51
oxygen|toluene                                                                         51
carbon dioxide|pentaerythritol tetrapentanoate                                         51
Na

In [ ]:
mask = binary_5plusperfile["mix_key"].isin(new_binary_mixture_counts.index)
binary_5plusperfile_high_mixture_count = binary_5plusperfile[mask]
print(f"Filtered shape: {binary_5plusperfile_high_mixture_count.shape}")
print(binary_5plusperfile_high_mixture_count["mix_key"].value_counts())

Filtered shape: (1117058, 73)
mix_key
germanium iron lithium oxide (ge2felio6)|germanium iron sodium oxide (ge2fenao6)    15941
ethanol|water                                                                        5116
carbon dioxide|methanol                                                              4546
triethylene glycol|water                                                             3635
methanol|water                                                                       3617
                                                                                    ...  
1-undecanol|acetonitrile                                                               51
cyclohexylamine|toluene                                                                51
2-methylbutan-2-ol|ethyl acetate                                                       51
oxygen|toluene                                                                         51
carbon dioxide|pentaerythritol tetrapentanoate                

In [ ]:
binary_5plusperfile_high_mixture_count.iloc[1]

material_id                                                 1__2
components                               methane, carbon dioxide
thermoml_fair_version                                     1.0.25
property                             Thermal conductivity, W/m/K
value                                                    0.02125
                                                ...             
Volume ratio of solute to solvent                            NaN
formula                                                         
comp_list                              [methane, carbon dioxide]
ncomp                                                          2
mix_key                                   carbon dioxide|methane
Name: 98, Length: 73, dtype: object

In [ ]:
# Split binary_5plusperfile_high_mixture_count in half and save to CSV
n = len(binary_5plusperfile_high_mixture_count)
mid = n // 2

df_half1 = binary_5plusperfile_high_mixture_count.iloc[:mid]
df_half2 = binary_5plusperfile_high_mixture_count.iloc[mid:]

file1 = "binary_5plusperfile_high_mixture_count_part1.csv"
file2 = "binary_5plusperfile_high_mixture_count_part2.csv"

df_half1.to_csv(file1, index=False)
df_half2.to_csv(file2, index=False)

print(f"Total rows: {n}")
print(f"Part 1 rows: {len(df_half1)}")
print(f"Part 2 rows: {len(df_half2)}")
print("Saved:")
print(f" - {file1}")
print(f" - {file2}")

Total rows: 1117058
Part 1 rows: 558529
Part 2 rows: 558529
Saved:
 - binary_5plusperfile_high_mixture_count_part1.csv
 - binary_5plusperfile_high_mixture_count_part2.csv
